# Занятие 28. Практика: признаки — РЕШЕНИЕ

**Только для преподавателя. Не выдавать студентам.**

Эталонные решения заданий 0–9 из `feature_engineering_practice.ipynb`.

---
## Дано: датасет

Синтетическая таблица объявлений (300 строк): площадь, комнаты, район, состояние, балкон, дата публикации, просмотры за месяц, доход района (есть пропуски) и **цена** — целевая переменная. Момент прогноза — `PREDICTION_DATE`.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 300

districts = rng.choice(['центр', 'спальный', 'пригород'], size=n, p=[0.3, 0.5, 0.2])
condition = rng.choice(['требует ремонта', 'среднее', 'хорошее'], size=n, p=[0.2, 0.5, 0.3])
area = rng.uniform(25, 130, n).round(0)
rooms = np.clip((area / 30).astype(int) + rng.integers(0, 2, n), 1, 5)
balcony = rng.integers(0, 2, n)
listing_date = pd.Timestamp('2026-01-01') + pd.to_timedelta(rng.integers(0, 180, n), unit='D')
views = np.expm1(rng.normal(4.0, 1.2, n)).round(0)

district_bonus = pd.Series(districts).map({'центр': 4.0, 'спальный': 1.0, 'пригород': 0.0}).values
condition_bonus = pd.Series(condition).map(
    {'требует ремонта': -1.5, 'среднее': 0.0, 'хорошее': 1.5}
).values
price = (0.13 * area + 0.02 * area * balcony + district_bonus
         + condition_bonus + rng.normal(0, 1.0, n)).round(1)

district_income = pd.Series(districts).map({'центр': 95.0, 'спальный': 60.0, 'пригород': 45.0}).values
district_income = district_income + rng.normal(0, 5, n).round(0)
missing_mask = rng.random(n) < 0.15
district_income[missing_mask] = np.nan

flats = pd.DataFrame({
    'площадь': area,
    'комнаты': rooms,
    'район': districts,
    'состояние': condition,
    'балкон': balcony,
    'дата': listing_date,
    'просмотры_за_месяц': views,
    'доход_района': district_income,
    'цена': price,
})

PREDICTION_DATE = pd.Timestamp('2026-07-01')
RANDOM_STATE = 42
print('Объектов:', len(flats))
flats.head()


---
## Задание 0. Постановка (~5 мин)

По мотивам п. 1 теории.

1. Создайте словарь `task_spec` с ключами `объект`, `цель`, `тип_задачи`, `момент_прогноза`.
2. В комментарии приведите один пример признака, который нельзя использовать из-за утечки данных.
   Пример: `цена_за_метр = цена / площадь` нельзя использовать, если `цена` — это цель, которую модель должна предсказать.
3. Коротко объясните, какая информация в этом признаке «подглядывает» в ответ.

**Критерий:** словарь заполнен; пример утечки назван и объяснён на уровне «из какого столбца подглянули ответ».

In [ ]:
task_spec = {
    'объект': 'объявление о продаже квартиры',
    'цель': 'цена, млн руб.',
    'тип_задачи': 'регрессия',
    'момент_прогноза': 'публикация объявления (PREDICTION_DATE для агрегатов)',
}
# Утечка: 'цена за метр' = цена / площадь — содержит целевую переменную.
task_spec


---
## Задание 1. Split (~7 мин)

По мотивам п. 0 теории (и занятия 25, п. 4).

1. Разбейте `flats` на `flats_train` и `flats_val` (70/30, `random_state=RANDOM_STATE`).
2. Выведите размеры.

**Критерий:** 210 / 90 объектов; дальше всё «обучаемое» — только по train.

In [ ]:
from sklearn.model_selection import train_test_split

flats_train, flats_val = train_test_split(flats, test_size=0.3, random_state=RANDOM_STATE)
print('train:', len(flats_train), '| validation:', len(flats_val))


---
## Задание 2. Числовые признаки (~10 мин)

По мотивам пп. 2–3 теории. В **обеих** таблицах (train и val):

1. `площадь_на_комнату` = площадь / комнаты.
2. `log_просмотры` = `np.log1p(просмотры_за_месяц)`.

**Критерий:** новые столбцы есть в train и val; значения без NaN.

In [ ]:
for table in (flats_train, flats_val):
    table['площадь_на_комнату'] = table['площадь'] / table['комнаты']
    table['log_просмотры'] = np.log1p(table['просмотры_за_месяц'])
flats_train[['площадь_на_комнату', 'log_просмотры']].describe().round(2)


---
## Задание 3. Признаки из даты (~10 мин)

По мотивам пп. 7–8 теории. В обеих таблицах:

1. `месяц` из даты; `месяц_sin`, `месяц_cos` (цикличность).
2. `дней_с_публикации` = `PREDICTION_DATE` − дата.

**Критерий:** `дней_с_публикации` ≥ 0 для всех строк.

In [ ]:
for table in (flats_train, flats_val):
    table['месяц'] = table['дата'].dt.month
    table['месяц_sin'] = np.sin(2 * np.pi * table['месяц'] / 12)
    table['месяц_cos'] = np.cos(2 * np.pi * table['месяц'] / 12)
    table['дней_с_публикации'] = (PREDICTION_DATE - table['дата']).dt.days
assert (flats_train['дней_с_публикации'] >= 0).all()
flats_train[['месяц', 'месяц_sin', 'месяц_cos', 'дней_с_публикации']].head().round(2)


---
## Задание 4. Категории (~12 мин)

По мотивам пп. 9–11 теории.

1. `OneHotEncoder(handle_unknown='ignore')` для `район`: `fit` на train, `transform` на train и val.
2. `OrdinalEncoder` для `состояние` с явным порядком: требует ремонта < среднее < хорошее; столбец `состояние_код` в обеих таблицах.

**Критерий:** кодировщики обучены только на train; в val нет ошибок.

In [ ]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

district_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
district_train = district_encoder.fit_transform(flats_train[['район']])
district_val = district_encoder.transform(flats_val[['район']])
district_cols = list(district_encoder.get_feature_names_out())

cond_encoder = OrdinalEncoder(categories=[['требует ремонта', 'среднее', 'хорошее']])
flats_train['состояние_код'] = cond_encoder.fit_transform(flats_train[['состояние']])
flats_val['состояние_код'] = cond_encoder.transform(flats_val[['состояние']])

for table, encoded in ((flats_train, district_train), (flats_val, district_val)):
    for j, col in enumerate(district_cols):
        table[col] = encoded[:, j]
print('One-hot столбцы:', district_cols)


---
## Задание 5. Пропуски (~8 мин)

По мотивам п. 12 теории.

1. `доход_пропущен` — индикатор 0/1 в обеих таблицах.
2. `доход_заполнен` — NaN заменены **медианой train**.

**Критерий:** медиана посчитана только по train; NaN не осталось.

In [ ]:
median_income = flats_train['доход_района'].median()
for table in (flats_train, flats_val):
    table['доход_пропущен'] = table['доход_района'].isna().astype(int)
    table['доход_заполнен'] = table['доход_района'].fillna(median_income)
assert flats_val['доход_заполнен'].notna().all()
print('Медиана train:', round(median_income, 1))


---
## Задание 6. Baseline-набор и kNN (~10 мин)

По мотивам пп. 15–16 теории.

1. Базовый набор: `['площадь', 'комнаты']`.
2. Соберите `Pipeline`: сначала `StandardScaler`, затем `KNeighborsRegressor(n_neighbors=5)`.
   `Pipeline` нужен, чтобы масштабирование обучалось на train и точно так же применялось к validation.
3. Обучите на train, посчитайте **MAE на validation** → `mae_base`.

**Критерий:** масштабирование обучается внутри `Pipeline` только на train.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def validation_mae(features):
    model = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsRegressor(n_neighbors=5)),
    ])
    model.fit(flats_train[features], flats_train['цена'])
    return mean_absolute_error(flats_val['цена'], model.predict(flats_val[features]))

base_features = ['площадь', 'комнаты']
mae_base = validation_mae(base_features)
print('MAE базовый набор:', round(mae_base, 3), 'млн')


---
## Задание 7. Добавляем группы признаков (~12 мин)

По мотивам п. 16 теории. Сравните на validation наборы (каждый = базовый + одна группа):

- `+ ['площадь_на_комнату', 'log_просмотры']`
- `+ ['состояние_код', 'балкон']`
- `+ one-hot районов`

**Критерий:** таблица «набор → MAE»; выводы о том, какая группа помогла.

In [ ]:
candidate_groups = {
    'числовые': ['площадь_на_комнату', 'log_просмотры'],
    'состояние+балкон': ['состояние_код', 'балкон'],
    'район (one-hot)': district_cols,
}
results = {'база': mae_base}
for name, group in candidate_groups.items():
    results[name] = validation_mae(base_features + group)
pd.Series(results).round(3).to_frame('MAE, млн')


---
## Задание 8. Комбинированный набор (~8 мин)

1. Соберите набор из базового + всех групп, которые **улучшили** MAE в задании 7.
2. Посчитайте `mae_final` на validation и сравните с `mae_base`.

**Критерий:** `mae_final` ≤ `mae_base`; состав набора обоснован результатами задания 7.

In [ ]:
final_features = base_features + candidate_groups['состояние+балкон'] + candidate_groups['район (one-hot)']
mae_final = validation_mae(final_features)
print('Финальный набор:', final_features)
print('MAE база:', round(mae_base, 3), '→ MAE финал:', round(mae_final, 3))


---
## Задание 9. Итог (~8 мин)

В markdown-ячейке ниже ответьте:

1. Какие группы признаков улучшили MAE, какие нет — и почему (гипотеза)?
2. Какой признак вы **не стали** строить из-за утечки?
3. Какие параметры обработки, посчитанные на train, нужно сохранить для завтрашних объявлений: кодировщики категорий, медиану для пропусков, масштабирование?

**Критерий:** ответы ссылаются на числа из заданий 7–8, а не «кажется».

**Эталонные тезисы:**

1. Состояние+балкон и район дают наибольший вклад — они напрямую входят в формулу цены; log-просмотры почти не помогают (просмотры не влияют на цену в этих данных).
2. «Цена за метр» — содержит цель; агрегат «средняя цена района по всем данным» — заглядывает в будущее и в validation.
3. Сохранить и применять: обученные на train кодировщики (`OneHotEncoder`, `OrdinalEncoder`), медиану дохода train, `StandardScaler` внутри pipeline; новые категории районов обрабатываются `handle_unknown='ignore'`.